# Lab 4 · Heavy job, Spark UI & resource prioritisation

Run a deliberately **skewed** aggregation, read the **Spark UI**, then a **tuned** version. Finally we look at how Fabric **prioritises resources** when a nightly ETL and an ad-hoc job compete.

> **Attach** the `lh_resident360` Lakehouse first.
> Set `DBX` to your Lab 1 mirror if it differs.

In [ ]:
from pyspark.sql import functions as F
DBX = "hpb_databricks_mirror.gold"
act = spark.table(f"{DBX}.daily_activity")
print("activity rows:", act.count())

## 1. Skewed job — everything forced through ONE partition
Run this, then open **Spark UI** (notebook status bar → **Spark UI**) → **Stages**. Note the single long task, shuffle read/write and any spill.

In [ ]:
heavy = (act.repartition(1)                                   # SKEW: single partition
         .groupBy("resident_id")
         .agg(F.avg("steps").alias("avg_steps"),
              F.expr("percentile_approx(steps, 0.9)").alias("p90_steps"),
              F.stddev("steps").alias("sd_steps")))
print("rows:", heavy.count())

## 2. Tuned job — Adaptive Query Execution + balanced partitions
Same result, healthier execution: compare task count and duration in the Spark UI.

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")          # AQE
tuned = (act.repartition(8, "resident_id")                    # 8 balanced partitions
         .groupBy("resident_id")
         .agg(F.avg("steps").alias("avg_steps"),
              F.expr("percentile_approx(steps, 0.9)").alias("p90_steps"),
              F.stddev("steps").alias("sd_steps")))
print("rows:", tuned.count())

## 3. Resource prioritisation — nightly ETL vs ad-hoc

On Fabric, **capacity (CU)** replaces cluster sizing. When a scheduled ETL and ad-hoc analysis run at once they share the same capacity, so a heavy ad-hoc job can starve the nightly load.

**How Fabric lets you prioritise / isolate:**
- **Custom (On-Demand) Spark pool + Environment** — give the nightly ETL its own pool so it isn't queued behind ad-hoc sessions.
- **Autoscale Billing for Spark** — move bursty ad-hoc Spark to dedicated serverless billed separately, so it never competes with the capacity that runs production ETL.
- **High-concurrency session sharing** — many light notebooks share one session to save CU for the jobs that matter.

**See it live:** open the facilitator's **Spark Monitoring** KQL dashboard (from the fabric-toolbox accelerator) to compare this heavy application's memory / CPU / shuffle / spill against a tuned run, and read its SparkLens recommendation on whether more resources would actually help.

> **Try it (optional):** re-run cell 1 while a neighbour runs a job on the same capacity, then watch both applications contend in **Monitor hub → Spark applications**.